# Tier 0 Repo Reality Check (Colab)

**Purpose:** Capture Tier 0 ground-truth build, run, and output behavior for Cholla in a Colab GPU environment.

**Target branch:** `dev`

**Environment:** Google Colab with GPU runtime

**Constraints:**
- Observation-only workflow
- No source modifications
- No patching examples
- No optimizations

**Expected outputs:** Reproducible command traces and observed ground truth only.


# Section 1 - Environment Detection

## 1.1 System Info

In [67]:
!uname -a
!cat /etc/os-release || true


Linux 92ef718cf41e 6.6.105+ #1 SMP Thu Oct  2 10:42:05 UTC 2025 x86_64 x86_64 x86_64 GNU/Linux
PRETTY_NAME="Ubuntu 22.04.5 LTS"
NAME="Ubuntu"
VERSION_ID="22.04"
VERSION="22.04.5 LTS (Jammy Jellyfish)"
VERSION_CODENAME=jammy
ID=ubuntu
ID_LIKE=debian
HOME_URL="https://www.ubuntu.com/"
SUPPORT_URL="https://help.ubuntu.com/"
BUG_REPORT_URL="https://bugs.launchpad.net/ubuntu/"
PRIVACY_POLICY_URL="https://www.ubuntu.com/legal/terms-and-policies/privacy-policy"
UBUNTU_CODENAME=jammy


In [68]:
ls

builds/             docs/        LICENSE.txt     python/              src/
cholla-tests-data/  examples/    Makefile        README.md            tools/
docker/             Jenkinsfile  pyproject.toml  scale_output_files/


## 1.2 GPU / CUDA Detection

In [69]:
!nvidia-smi || true
!nvcc --version || true
!which nvcc || true


Fri Feb 13 20:06:17 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1.3 MPI Detection

In [70]:
!mpirun --version || true
!which mpirun || true
!mpicxx --version || true
!which mpicxx || true


mpirun (Open MPI) 4.1.2

Report bugs to http://www.open-mpi.org/community/help/
/usr/bin/mpirun
g++ (Ubuntu 11.4.0-1ubuntu1~22.04.2) 11.4.0
Copyright (C) 2021 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.

/usr/bin/mpicxx


In [71]:
%%bash
set -e
if ! command -v mpirun >/dev/null 2>&1 || ! command -v mpicxx >/dev/null 2>&1; then
  apt-get update
  apt-get install -y openmpi-bin libopenmpi-dev
else
  echo "MPI detected; skipping install."
fi

mpirun --version || true
which mpirun || true
mpicxx --version || true
which mpicxx || true


MPI detected; skipping install.
mpirun (Open MPI) 4.1.2

Report bugs to http://www.open-mpi.org/community/help/
/usr/bin/mpirun
g++ (Ubuntu 11.4.0-1ubuntu1~22.04.2) 11.4.0
Copyright (C) 2021 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.

/usr/bin/mpicxx


## 1.4 HDF5 Detection

In [72]:
!which h5cc || true
!h5cc -showconfig || true


/usr/bin/h5cc
	    SUMMARY OF THE HDF5 CONFIGURATION

General Information:
-------------------
                   HDF5 Version: 1.10.7
                  Configured on: Wed, 08 Dec 2021 23:33:27 +0000
                  Configured by: Debian
                    Host system: x86_64-pc-linux-gnu
              Uname information: Debian
                       Byte sex: little-endian
             Installation point: /usr
		    Flavor name: serial

Compiling Options:
------------------
                     Build Mode: production
              Debugging Symbols: no
                        Asserts: no
                      Profiling: no
             Optimization Level: high

Linking Options:
----------------
                      Libraries: static, shared
  Statically Linked Executables: 
                        LDFLAGS: -Wl,-Bsymbolic-functions -flto=auto -Wl,-z,relro
                     H5_LDFLAGS: -Wl,--version-script,$(top_srcdir)/debian/map_serial.ver
                     AM_LDFLAGS: 
    

## 1.5 Toolchain

In [73]:
!gcc --version
!g++ --version
!make --version
!cmake --version || true
!python3 --version


gcc (Ubuntu 11.4.0-1ubuntu1~22.04.2) 11.4.0
Copyright (C) 2021 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.

g++ (Ubuntu 11.4.0-1ubuntu1~22.04.2) 11.4.0
Copyright (C) 2021 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.

GNU Make 4.3
Built for x86_64-pc-linux-gnu
Copyright (C) 1988-2020 Free Software Foundation, Inc.
License GPLv3+: GNU GPL version 3 or later <http://gnu.org/licenses/gpl.html>
This is free software: you are free to change and redistribute it.
There is NO WARRANTY, to the extent permitted by law.
cmake version 3.31.10

CMake suite maintained and supported by Kitware (kitware.com/cmake).
Python 3.12.12


# Section 2 - Clone + Checkout

In [74]:
!git clone https://github.com/JinchuLi2002/cholla.git
%cd cholla
!git checkout dev
!git pull --ff-only
!git rev-parse HEAD
!git status --porcelain


Cloning into 'cholla'...
remote: Enumerating objects: 29062, done.
remote: Counting objects: 100% (912/912), done.
remote: Compressing objects: 100% (241/241), done.
remote: Total 29062 (delta 752), reused 703 (delta 669), pack-reused 28150 (from 3)
Receiving objects: 100% (29062/29062), 53.82 MiB | 17.76 MiB/s, done.
Resolving deltas: 100% (21242/21242), done.
/content/cholla/cholla/cholla/cholla
Branch 'dev' set up to track remote branch 'dev' from 'origin'.
Switched to a new branch 'dev'
Already up to date.
1682f0283a6d20cc3b74e6df1eff43eb535af0b4


In [75]:
!ls ../

builds		   docker    Jenkinsfile  pyproject.toml  scale_output_files
cholla		   docs      LICENSE.txt  python	  src
cholla-tests-data  examples  Makefile	  README.md	  tools


In [76]:
!git submodule update --init --recursive
!git submodule status


Submodule 'cholla-tests-data' (https://github.com/cholla-hydro/cholla-tests-data.git) registered for path 'cholla-tests-data'
Cloning into '/content/cholla/cholla/cholla/cholla/cholla-tests-data'...
Submodule path 'cholla-tests-data': checked out 'aceebfdd33b3a86b39ec354bfbd630284201b66b'
 aceebfdd33b3a86b39ec354bfbd630284201b66b cholla-tests-data (heads/main)


# Section 3 - Build System Detection

In [77]:
!ls -la
!ls -la builds || true
!ls -la Makefile || true
!ls -la CMakeLists.txt || true


total 116
drwxr-xr-x 13 root root 4096 Feb 13 20:06 .
drwxr-xr-x 14 root root 4096 Feb 13 20:06 ..
drwxr-xr-x  2 root root 4096 Feb 13 20:06 builds
drwxr-xr-x  4 root root 4096 Feb 13 20:06 cholla-tests-data
-rw-r--r--  1 root root 6713 Feb 13 20:06 .clang-format
-rw-r--r--  1 root root 7549 Feb 13 20:06 .clang-tidy
drwxr-xr-x  4 root root 4096 Feb 13 20:06 docker
drwxr-xr-x  4 root root 4096 Feb 13 20:06 docs
drwxr-xr-x  6 root root 4096 Feb 13 20:06 examples
drwxr-xr-x  9 root root 4096 Feb 13 20:06 .git
-rw-r--r--  1 root root  671 Feb 13 20:06 .git-blame-ignore-revs
drwxr-xr-x  3 root root 4096 Feb 13 20:06 .github
-rw-r--r--  1 root root 1088 Feb 13 20:06 .gitignore
-rw-r--r--  1 root root  119 Feb 13 20:06 .gitmodules
-rw-r--r--  1 root root 6224 Feb 13 20:06 Jenkinsfile
-rw-r--r--  1 root root 1080 Feb 13 20:06 LICENSE.txt
-rw-r--r--  1 root root 7405 Feb 13 20:06 Makefile
-rw-r--r--  1 root root 1125 Feb 13 20:06 .pre-commit-config.yaml
-rw-r--r--  1 root root 2818 Feb 13 20:06

In [78]:
!grep -R -n "TYPE=cosmology" . | head -n 50
!grep -R -n "make.type.cosmology" builds | head -n 50


# Section 4 - Cosmology Build

## 4.1 Host Detection Behavior

In [79]:
!hostname
!bash builds/machine.sh || true


92ef718cf41e
Using default hostname, expecting make.host.92ef718cf41e
92ef718cf41e


In [80]:
%%bash
set -o pipefail
NVCC_PATH="$(command -v nvcc || true)"
if [ -n "$NVCC_PATH" ]; then
  export CUDA_ROOT="${CUDA_ROOT:-$(dirname "$(dirname "$NVCC_PATH")")}" 
fi
export MPI_ROOT="${MPI_ROOT:-/usr}"
export HDF5_ROOT="${HDF5_ROOT:-/usr}"

MPI_INC="$(mpicxx --showme:compile 2>/dev/null | grep -oE -- '-I[^ ]+' | paste -sd' ' - || true)"
if [ -z "$MPI_INC" ]; then
  MPI_INC="$(mpicxx -show 2>/dev/null | grep -oE -- '-I[^ ]+' | paste -sd' ' - || true)"
fi

H5_SHOW="$(h5cc -show 2>/dev/null || true)"
H5_INC="$(echo "$H5_SHOW" | grep -oE -- '-I[^ ]+' | paste -sd' ' - || true)"
H5_LIBDIR="$(echo "$H5_SHOW" | grep -oE -- '-L[^ ]+' | paste -sd' ' - || true)"

if [ -n "$MPI_INC" ]; then
  export CXXFLAGS="${CXXFLAGS:-} $MPI_INC"
  export GPUFLAGS="${GPUFLAGS:-} $MPI_INC"
fi
if [ -n "$H5_INC" ]; then
  export CXXFLAGS="${CXXFLAGS:-} $H5_INC"
  export GPUFLAGS="${GPUFLAGS:-} $H5_INC"
fi
if [ -n "$H5_LIBDIR" ]; then
  export LIBS="${LIBS:-} $H5_LIBDIR"
fi

make TYPE=cosmology -j2 || CHOLLA_MACHINE=github make TYPE=cosmology -j2


builds/prereq.sh build github
mpicxx  -I/usr/lib/x86_64-linux-gnu/openmpi/include -I/usr/lib/x86_64-linux-gnu/openmpi/include/openmpi -I/usr/include/hdf5/serial -Ofast -std=c++17  -DMPI_CHOLLA -DPRECISION=2 -DHLLC -DSIMPLE -DPPMP -DDENSITY_FLOOR -DTEMPERATURE_FLOOR -DOUTPUT -DHDF5  -DGRAVITY -DPARIS -DGRAVITY_GPU -DGRAVITY_5_POINTS_GRADIENT -DPARALLEL_OMP -DN_OMP_THREADS=7  -DPARTICLES -DPARTICLES_GPU -DPARTICLE_IDS -DSINGLE_PARTICLE_MASS -DPARALLEL_OMP -DN_OMP_THREADS=7 -DCOSMOLOGY -DAVERAGE_SLOW_CELLS -DDE -DPRINT_INITIAL_STATS -DN_OUTPUT_COMPLETE=1 -DPARIS_5PT -DGIT_HASH='"1682f0283a6d20cc3b74e6df1eff43eb535af0b4"' -DMACRO_FLAGS='"-DMPI_CHOLLA -DPRECISION=2 -DHLLC -DSIMPLE -DPPMP -DDENSITY_FLOOR -DTEMPERATURE_FLOOR -DOUTPUT -DHDF5  -DGRAVITY -DPARIS -DGRAVITY_GPU -DGRAVITY_5_POINTS_GRADIENT -DPARALLEL_OMP -DN_OMP_THREADS=7  -DPARTICLES -DPARTICLES_GPU -DPARTICLE_IDS -DSINGLE_PARTICLE_MASS -DPARALLEL_OMP -DN_OMP_THREADS=7 -DCOSMOLOGY -DAVERAGE_SLOW_CELLS -DDE -DPRINT_INITIAL_STATS -D

Using default hostname, expecting make.host.92ef718cf41e
Using default hostname, expecting make.host.92ef718cf41e
Makefile:6: builds/make.host.92ef718cf41e: No such file or directory
make: *** No rule to make target 'builds/make.host.92ef718cf41e'.  Stop.
src/gravity/gravity_restart.cpp: In member function ‘void Grav3D::Read_Restart_HDF5(Parameters*, int)’:
src/gravity/gravity_restart.cpp:23:26: warning: ‘_gravity.h5.’ directive writing 12 bytes into a region of size between 0 and 2047 [-Wformat-overflow=]
   23 |   sprintf(filename, "%s%d_gravity.h5.%d", dirname, nfile, procID);
      |                          ^~~~~~~~~~~~
In file included from /usr/include/stdio.h:894,
                 from /usr/include/c++/11/cstdio:42,
                 from src/gravity/gravity_restart.cpp:4:
/usr/include/x86_64-linux-gnu/bits/stdio2.h:38:34: note: ‘__builtin___sprintf_chk’ output between 15 and 2082 bytes into a destination of size 2048
   38 |   return __builtin___sprintf_chk (__s, __USE_FORTIFY_

# Section 5 - Binary Discovery

In [81]:
!find . -maxdepth 4 -type f -perm -111 | sort


./bin/cholla.cosmology.github
./builds/check.sh
./builds/machine.sh
./builds/prereq.sh
./builds/run_tests.sh
./builds/setup.birch.cce.sh
./builds/setup.c3po.gcc.sh
./builds/setup.crc.gcc.sh
./builds/setup.frontier.cce.sh
./builds/setup.github.gcc.sh
./builds/setup.lux.sh
./builds/setup.poplar.aomp.sh
./builds/setup.poplar.cce+hip.sh
./builds/setup.shamrock.sh
./builds/setup.spock.cce.sh
./builds/setup.summit.gcc.sh
./builds/setup.summit.xl.sh
./builds/setup.vista.sh
./examples/scripts/acrun-hydro.sh
./examples/scripts/acrun-paris-hipfft.sh
./examples/scripts/acrun-paris-pfft.sh
./examples/scripts/acrun-paris.sh
./examples/scripts/acrun-paris-sor.sh
./examples/scripts/acrun-sor.sh
./examples/scripts/arun-hydro.sh
./examples/scripts/arun-paris-hipfft.sh
./examples/scripts/arun-paris-pfft.sh
./examples/scripts/arun-paris.sh
./examples/scripts/arun-paris-sor.sh
./examples/scripts/arun-paris-sphere.sh
./examples/scripts/arun-sor.sh
./examples/scripts/nrun-paris-cufft.sh
./examples/scripts/n

Record from the command output:
- Relative binary path
- Binary filename
- Whether multiple binaries exist


# Section 6 - Minimal Cosmology Run Attempt

## 6.1 Identify Candidate Example

In [82]:
!grep -R -n "Adiabatic_Expansion" examples
!ls examples/3D


examples/3D/Adiabatic_Expansion.txt:19:init=Adiabatic_Expansion
Adiabatic_Expansion.txt		      float32_sound_wave.txt
advecting_field_loop.txt	      isolated_star_particle.txt
alfven_wave.txt			      KH_res_ind_3D.txt
Brio_and_Wu.txt			      mhd_blast.txt
CAAR_FOM_Cosmo_512.txt		      mhd_contact_wave.txt
CAAR_FOM_Cosmo_8.txt		      Noh_3D.txt
circularly_polarized_alfven_wave.txt  orszag_tang_vortex.txt
constant.txt			      Ryu_and_Jones_1a.txt
Cosmological_cool_UV_50Mpc.txt	      Ryu_and_Jones_4d.txt
Cosmological_dm_50Mpc.txt	      Scaling_128.txt
Cosmological_hydro_256_50Mpc.txt      Scaling_256.txt
Cosmological_Scaling_128.txt	      slow_magnetosonic.txt
Cosmological_Scaling_256.txt	      sod256.txt
Cosmological_Scaling_384.txt	      sod.txt
Dai_and_Woodward.txt		      sound_wave.txt
disk_particle.txt		      Spherical_Collapse.txt
disk.txt			      Spherical_Overpressure.txt
Einfeldt_Strong_Rarefaction.txt       Uniform.txt
fast_magnetosonic.txt		      Zeldovich_Pancake.txt


In [83]:
!sed -n '1,200p' examples/3D/Adiabatic_Expansion.txt


#
# Parameter File for the 3D Adiabatic Expansion test.
#

######################################
# number of grid cells in the x dimension
nx=256
# number of grid cells in the y dimension
ny=32
# number of grid cells in the z dimension
nz=32
# output time
tout=1000
# how often to output
outstep=1000
# value of gamma
gamma=1.66666667
# name of initial conditions
init=Adiabatic_Expansion
#Cosmological Parameters 
Init_redshift=20.0
#Init_redshift=0.998294693667
H0=50.0
Omega_M=1.0
Omega_L=0.0
Omega_b=1.0
temperature_floor=1.0e-2
scale_outputs_file=scale_output_files/outputs_zeldovich_grav4.txt
# domain properties
xmin=0.0
ymin=0.0
zmin=0.0
xlen=64000.0
ylen=8000.0
zlen=8000.0
# type of boundary conditions
xl_bcnd=1
xu_bcnd=1
yl_bcnd=1
yu_bcnd=1
zl_bcnd=1
zu_bcnd=1
# path to output directory
indir=ics/
outdir=./


In [84]:
!grep -n "/home\|/raid\|/gpfs\|/data" examples/3D/*.txt || true


examples/3D/Cosmological_cool_UV_50Mpc.txt:44:#indir=/home/brvillas/simulations/256_cool_uv_50Mpc/ics_enzo/
examples/3D/Cosmological_cool_UV_50Mpc.txt:45:#outdir=/home/brvillas/simulations/256_cool_uv_50Mpc/output_files/
examples/3D/Cosmological_cool_UV_50Mpc.txt:46:indir=/raid/bruno/data/cosmo_sims/cholla_pm/256_cool_uv_50Mpc/ics_enzo/
examples/3D/Cosmological_cool_UV_50Mpc.txt:47:outdir=/raid/bruno/data/cosmo_sims/cholla_pm/256_cool_uv_50Mpc/
examples/3D/Cosmological_cool_UV_50Mpc.txt:48:#indir=/gpfs/alpine/scratch/bvilasen/ast149/cosmo_256/ics/
examples/3D/Cosmological_cool_UV_50Mpc.txt:49:#outdir=/gpfs/alpine/scratch/bvilasen/ast149/cosmo_256/output_snapshots/
examples/3D/Cosmological_dm_50Mpc.txt:42:indir=/data/groups/comp-astro/bruno/cosmo_sims/256_dm_50Mpc/ics/
examples/3D/Cosmological_dm_50Mpc.txt:43:outdir=/data/groups/comp-astro/bruno/cosmo_sims/256_dm_50Mpc/output_files/
examples/3D/Cosmological_dm_50Mpc.txt:44:#indir=/home/bruno/Desktop/hard_drive_1/data/cosmo_sims/cholla_p

In [85]:
!ls examples/3D/scale_output_files || true
!ls examples/3D/ics || true


ls: cannot access 'examples/3D/scale_output_files': No such file or directory
ls: cannot access 'examples/3D/ics': No such file or directory


In [88]:
!git status


On branch dev
Your branch is up to date with 'origin/dev'.

nothing to commit, working tree clean


## 6.2 Run Attempt

In [86]:
%%bash
set -u
mkdir -p _tier0_tmp_out

BIN_PATH=""
if [ -x ./bin/cholla ]; then
  BIN_PATH=./bin/cholla
else
  BIN_PATH=$(find . -maxdepth 4 -type f -perm -111 | sort | head -n 1 || true)
fi

if [ -z "$BIN_PATH" ]; then
  echo "No executable binary found. Skipping run attempt."
  exit 0
fi

echo "Selected binary path: $BIN_PATH"

if command -v mpirun >/dev/null 2>&1; then
  mpirun -n 1 "$BIN_PATH" examples/3D/Adiabatic_Expansion.txt || "$BIN_PATH" examples/3D/Adiabatic_Expansion.txt
else
  "$BIN_PATH" examples/3D/Adiabatic_Expansion.txt
fi


Selected binary path: ./bin/cholla.cosmology.github
Git Commit Hash = 1682f0283a6d20cc3b74e6df1eff43eb535af0b4
Macro Flags     = -DMPI_CHOLLA -DPRECISION=2 -DHLLC -DSIMPLE -DPPMP -DDENSITY_FLOOR -DTEMPERATURE_FLOOR -DOUTPUT -DHDF5  -DGRAVITY -DPARIS -DGRAVITY_GPU -DGRAVITY_5_POINTS_GRADIENT -DPARALLEL_OMP -DN_OMP_THREADS=7  -DPARTICLES -DPARTICLES_GPU -DPARTICLE_IDS -DSINGLE_PARTICLE_MASS -DPARALLEL_OMP -DN_OMP_THREADS=7 -DCOSMOLOGY -DAVERAGE_SLOW_CELLS -DDE -DPRINT_INITIAL_STATS -DN_OUTPUT_COMPLETE=1 -DPARIS_5PT -DGIT_HASH=1682f0283a6d20cc3b74e6df1eff43eb535af0b4
Parameter values:  nx = 256, ny = 32, nz = 32, tout = 1000.000000, init = Adiabatic_Expansion, boundaries = 1 1 1 1 1 1
Output directory:  ./

Creating Log File: run_output.log 

nproc_x 1 nproc_y 1 nproc_z 1
Allocating MPI communication buffers on GPU (nx = 24576, ny = 202752, nz = 253440).
Allocating MPI communication buffers on GPU for particle transfers ( N_Particles: 139392 ).
Allocating MPI communication buffers on Host

--------------------------------------------------------------------------
mpirun has detected an attempt to run as root.

Running as root is *strongly* discouraged as any mistake (e.g., in
defining TMPDIR) or bug can result in catastrophic damage to the OS
file system, leaving your system in an unusable state.

We strongly suggest that you run mpirun as a non-root user.

You can override this protection by adding the --allow-run-as-root option
to the cmd line or by setting two environment variables in the following way:
the variable OMPI_ALLOW_RUN_AS_ROOT=1 to indicate the desire to override this
protection, and OMPI_ALLOW_RUN_AS_ROOT_CONFIRM=1 to confirm the choice and
add one more layer of certainty that you want to do so.
We reiterate our advice against doing so - please proceed at your own risk.
--------------------------------------------------------------------------


CalledProcessError: Command 'b'set -u\nmkdir -p _tier0_tmp_out\n\nBIN_PATH=""\nif [ -x ./bin/cholla ]; then\n  BIN_PATH=./bin/cholla\nelse\n  BIN_PATH=$(find . -maxdepth 4 -type f -perm -111 | sort | head -n 1 || true)\nfi\n\nif [ -z "$BIN_PATH" ]; then\n  echo "No executable binary found. Skipping run attempt."\n  exit 0\nfi\n\necho "Selected binary path: $BIN_PATH"\n\nif command -v mpirun >/dev/null 2>&1; then\n  mpirun -n 1 "$BIN_PATH" examples/3D/Adiabatic_Expansion.txt || "$BIN_PATH" examples/3D/Adiabatic_Expansion.txt\nelse\n  "$BIN_PATH" examples/3D/Adiabatic_Expansion.txt\nfi\n'' returned non-zero exit status 1.

# Section 7 - Output Directory Inspection

In [ ]:
!find . -maxdepth 5 -type f | sort | head -n 100


Identify from observed output:
- Output directory path
- File naming patterns
- Whether files are sharded
- Example HDF5 filenames


# Section 8 - Code Inspection for Run Semantics

In [ ]:
!grep -R -n "tout" src | head -n 50
!grep -R -n "outstep" src | head -n 50
!grep -R -n "n_steps_limit" src | head -n 50
!grep -R -n "scale_outputs_file" src | head -n 50
!grep -R -n "End_redshift" src | head -n 50


In [ ]:
!nl -ba src/main.cpp | sed -n '200,350p'


# Section 9 - Final Markdown Summary (Observed Results)

Fill this summary only with observed results from this notebook run:

## A) Build result (success/failure) and exact command used
- Build result: `<success|failure>`
- Exact command that produced final result: `<paste exact command>`
- Notes (error/success evidence): `<paste key observed lines>`

## B) Binary path(s)
- Primary binary path: `<relative path>`
- Additional executable paths (if any): `<list>`

## C) Run attempt result
- Command path used: `<binary path + param file>`
- Launch mode: `<mpirun -n 1 | direct>`
- Result: `<success|failure>`
- Key observed stdout/stderr lines: `<paste>`

## D) Output directory and file naming pattern
- Output root directory: `<path>`
- Naming pattern(s): `<pattern>`
- Sharded outputs: `<yes|no|unknown>`
- Example HDF5 filenames: `<list>`

## E) One-step run semantics (config keys + source references)
- Relevant config keys observed: `<keys>`
- Source references: `<file:line>`
- Interpreted one-step semantics from code: `<observed behavior only>`

## F) Concrete blockers for Tier 1
- Blocker 1: `<specific blocker + evidence>`
- Blocker 2: `<specific blocker + evidence>`
- Blocker 3: `<specific blocker + evidence>`
